# Task 2 — Profile before you touch (8 columns, staging only)

The staging table is **1,284 rows**: shaped and renamed, nothing transformed.
This notebook profiles all eight contract columns **before** a single value is
changed. For each column: blanks, distinct raw values, rows that **fail the
stated rule**, and rows that merely **need a text transform** — a correct
value in the wrong form.

Convention: distinct raw values exclude blanks (blanks have their own column).
A row can fail two rules, and a row that fails a rule is not automatically a
row you reject — keeping those two ideas apart is most of the marks.

Run all cells top to bottom from the `Task6/` folder.


In [1]:
import importlib.util
from pathlib import Path
import pandas as pd

NB_DIR = Path.cwd()
DATA = NB_DIR / "data"
print("notebook dir:", NB_DIR)
print("data dir exists:", DATA.exists())

# The pipeline script's helpers, loaded from the file sitting next to this
# notebook — the same strict rules as W6_su92-bsdsm-f23-018.py: plain-decimal
# numbers only, DD/MM/YYYY-or-ISO dates, positional extract (headers differ
# across the three files, so columns are read by position, not by name).
_spec = importlib.util.spec_from_file_location("w6pipe", NB_DIR / "W6_su92-bsdsm-f23-018.py")
w6 = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(w6)
CONTRACT, PLAIN, STATUS_MAP = w6.CONTRACT, w6.PLAIN, w6.STATUS_MAP
norm_id, plain_number, parse_date = w6.norm_id, w6.plain_number, w6.parse_date
extract_csv, extract_rail = w6.extract_csv, w6.extract_rail
find_source_files, mode_key, bizkey = w6.find_source_files, w6.mode_key, w6.bizkey
print("pipeline helpers loaded from W6_su92-bsdsm-f23-018.py")

notebook dir: /home/hatch/workspace/data-warehouse/week6/Task6
data dir exists: True
pipeline helpers loaded from W6_su92-bsdsm-f23-018.py


In [2]:
from collections import Counter

marine_path, rail_path, road_path = find_source_files(DATA / "march")
staging, artifacts = [], []
for path, kind in ((marine_path, "csv"), (rail_path, "xlsx"), (road_path, "csv")):
    rows, arts = extract_csv(path, path.name) if kind == "csv" else extract_rail(path, path.name)
    staging.extend(rows)
    artifacts.extend(arts)
    print(f"  {path.name}: {len(rows)} rows")
for a in artifacts:
    print(f"  extract artifact (NOT a row): {a[0]} line {a[1]} — {a[2]}")
stg = pd.DataFrame(staging)[CONTRACT + ["SourceFile", "SourceLine", "SourceRow"]]
print("per-file rows:", dict(Counter(stg["SourceFile"])))
print("staging rows:", len(stg))
assert len(stg) == 1284, "staging must be 1,284 rows"


  marine_march.csv: 261 rows
  rail_march.xlsx: 315 rows
  road_march.csv: 708 rows
  extract artifact (NOT a row): road_march.csv line 710 — blank line
  extract artifact (NOT a row): road_march.csv line 711 — TOTAL footer
per-file rows: {'marine_march.csv': 261, 'rail_march.xlsx': 315, 'road_march.csv': 708}
staging rows: 1284


In [3]:
alias = {r["RawSpelling"].strip(): int(r["CityKey"]) for r in
         pd.read_csv(DATA / "alias_lookup.csv", dtype=str, keep_default_na=False).to_dict("records")}
dim_city = pd.read_csv(DATA / "dims" / "DimCity.csv", dtype=str, keep_default_na=False)
city_name = {int(r["CityKey"]): r["CityName"] for r in dim_city.to_dict("records")}
route = {(int(r["OriginCityKey"]), int(r["DestCityKey"])): int(r["RouteKey"]) for r in
         pd.read_csv(DATA / "dims" / "DimRoute.csv", dtype=str, keep_default_na=False).to_dict("records")}
dim_status = pd.read_csv(DATA / "dims" / "DimStatus.csv", dtype=str, keep_default_na=False)
status_names = {int(r["StatusKey"]): r["StatusName"] for r in dim_status.to_dict("records")}
print(f"alias spellings: {len(alias)}; cities: {len(city_name)}; routes: {len(route)}; statuses: {status_names}")

alias spellings: 23; cities: 15; routes: 21; statuses: {1: 'Delivered', 2: 'In Transit', 3: 'Delayed', -1: 'Unknown'}


### The rules (from the brief)

| Column | Rule a row must satisfy |
|---|---|
| ShipmentID | present, unique once trimmed and upper-cased |
| ShipDate | parses as a real date inside March 2026 |
| Origin | resolves to a city in alias_lookup |
| Destination | resolves to a city in alias_lookup |
| WeightKG | numeric and > 0 |
| FreightCostPKR | numeric (a negative is a credit note, not a fault) |
| StatusRaw | matches a DimStatus member once trimmed and cased |
| CustomerID | resolves to exactly one business in the registry |

In [4]:
# Blanks / distinct raw / rule failures / text-transform-only rows, all computed.
def is_blank(v):
    return (v or "").strip() == ""

def distinct_raw(s):
    return s.map(lambda v: None if is_blank(v) else v).nunique()

def fails_sid(s):
    seen, fails = set(), 0
    for v in s:
        n = norm_id(v)
        if n == "" or n in seen:
            fails += 1
        else:
            seen.add(n)
    return fails

def trans_sid(s):
    return int(s.map(lambda v: v != norm_id(v)).sum())

def fails_date(s):
    bad = s.map(lambda v: parse_date(v) is None)
    parsed = s.map(parse_date)
    outside = parsed.map(lambda d: d is not None and not (d.year == 2026 and d.month == 3))
    return int((bad | outside).sum())

def fails_city(s):
    return int(s.map(lambda v: alias.get((v or "").strip()) is None).sum())

def trans_city(s):
    def tr(v):
        k = alias.get((v or "").strip())
        return k is not None and v != city_name[k]
    return int(s.map(tr).sum())

def fails_weight(s):
    text = s.map(lambda v: plain_number(v) is None and (v or "").strip() != "")
    neg = s.map(lambda v: plain_number(v) is not None and plain_number(v) < 0)
    return int((text | neg).sum())

def fails_cost(s):
    return int(s.map(lambda v: plain_number(v) is None).sum())

def fails_status(s):
    return int(s.map(lambda v: STATUS_MAP.get((v or "").strip().upper()) is None).sum())

def trans_status(s):
    def tr(v):
        m = STATUS_MAP.get((v or "").strip().upper())
        return m is not None and v != m[0]
    return int(s.map(tr).sum())

reg_ids = set(pd.read_csv(DATA / "customers_master.csv", dtype=str, keep_default_na=False)["CustomerID"])

rows = [
    ("ShipmentID", stg["ShipmentID"], fails_sid, trans_sid),
    ("ShipDate", stg["ShipDate"], fails_date, lambda s: "—"),
    ("Origin", stg["Origin"], fails_city, trans_city),
    ("Destination", stg["Destination"], fails_city, trans_city),
    ("WeightKG", stg["WeightKG"], fails_weight, lambda s: "—"),
    ("FreightCostPKR", stg["FreightCostPKR"], fails_cost, lambda s: "—"),
    ("StatusRaw", stg["StatusRaw"], fails_status, trans_status),
    ("CustomerID", stg["CustomerID"], lambda s: int(s.map(lambda v: v not in reg_ids).sum()), lambda s: "—"),
]
prof = pd.DataFrame([{"Column": name,
                      "Blanks": int(col.map(is_blank).sum()),
                      "Distinct raw": distinct_raw(col),
                      "Rule failures": f(col),
                      "Text-transform-only": t(col)}
                     for name, col, f, t in rows])
print(prof.to_string(index=False))
total_fails = int(prof["Rule failures"].sum())
print(f"\nrule-failure counts sum to: {total_fails}")

        Column  Blanks  Distinct raw  Rule failures Text-transform-only
    ShipmentID       4          1232             64                 712
      ShipDate       0            67              6                   —
        Origin       0            18             14                 570
   Destination       0            19              0                 189
      WeightKG       0            78             14                   —
FreightCostPKR       0           378              0                   —
     StatusRaw       0            25              7                 980
    CustomerID       0            46              0                   —

rule-failure counts sum to: 105


In [5]:
expected = {"ShipmentID": (4, 1232, 64, 712), "ShipDate": (0, 67, 6, "—"),
            "Origin": (0, 18, 14, 570), "Destination": (0, 19, 0, 189),
            "WeightKG": (0, 78, 14, "—"), "FreightCostPKR": (0, 378, 0, "—"),
            "StatusRaw": (0, 25, 7, 980), "CustomerID": (0, 46, 0, "—")}
for _, r in prof.iterrows():
    got = (int(r["Blanks"]), int(r["Distinct raw"]), int(r["Rule failures"]), r["Text-transform-only"])
    assert got == expected[r["Column"]], f"{r['Column']}: got {got}, expected {expected[r['Column']]}"
assert total_fails == 105, f"self-check: failure counts must sum to 105, got {total_fails}"
print("SELF-CHECK PASS — eight rule-failure counts sum to 105 (not 84, not 24)")

# Analysis
# Profile before transforms because transforms destroy the evidence of what
# arrived — after trim/upper/case/unit fixes you can no longer count the
# faults, prove the 1284 = 1200 + 24 + 60 identity, or tell a fault from a
# form; the profile is the baseline the audit, the reject contract and the
# reconciliation all read from, and transforms themselves can create or hide
# duplicates (trailing-space copies) and move totals (rail tonnes).
#
# Best shape = FreightCostPKR (0 rule failures, 0 transforms needed; its 3
# negatives are legitimate credit notes, not faults). Distinct count most
# overstates = StatusRaw (25 distinct raw strings for 4 real values — 3
# statuses + Unknown — because case and stray spaces multiply spellings ~6x;
# the column looks filthy, 980 rows need a case/trim transform, yet only 7
# rows actually fail, all from 3 truly unknown strings). The difference:
# distinct counts FORMS (spellings), fails counts RULE violations
# (substance) — a column can look dirty and be nearly clean, while WeightKG
# looks numeric and hides 14 failures.

SELF-CHECK PASS — eight rule-failure counts sum to 105 (not 84, not 24)
